# 06 · Enterprise RAG — permission-aware BFSI policy retrieval

## Business situation

**Question:** A Northstar Bank risk analyst needs guidance on a credit decision. That analyst must not see confidential AML compliance documents, a different bank's KYC documents, or expired policies.

This lab uses **fictional policies**, real LangChain `Document`, `RecursiveCharacterTextSplitter` and **ChromaDB vector search**. The default embedding is **deterministic lexical hashing**, not a neural embedding model, so CI does not require an Ollama download.

**Security principle:** permissions must be enforced before vector similarity search, not applied only after global nearest-neighbor retrieval.

```text
Synthetic JSONL (tenant, group, validity, source, version)
                  |
           LangChain Document
                  |
        RecursiveCharacterTextSplitter
                  |
        Chroma vector index + metadata
                  |
Authenticated identity (simulated) -> tenant, role, as-of date filter
                  |                           |
              question ----------> scoped similarity search
                                               |
                                        cited evidence
```

**Predict:** Why is filtering `top-k` results *after* searching all tenants unsafe? A global top-k may eliminate all permitted candidates and can create cross-tenant exposure in logs, rerankers or prompts.

In [ ]:
from datetime import date
from RAG.enterprise.langchain_bfsi import (
    Principal, load_policies, chunk_policies, build_store,
    OfflineLexicalEmbeddings, scoped_filter, authorized_search,
    ask_policy, load_evaluation, evaluate_access_and_retrieval
)
TODAY = date(2026, 10, 9)
policies = load_policies()
print("Loaded fictional policy records:", len(policies))
for p in policies[:4]:
    print(p.metadata["policy_id"], "tenant=", p.metadata["tenant_id"],
          "group=", p.metadata["access_group"], "version=", p.metadata["version"])

## Trace LangChain documents into a real vector store

`Document(page_content, metadata)` separates searchable text and scope metadata. Splitting preserves the metadata. By default Chroma is local and in-memory. A production system would use isolated persistent/encrypted stores and explicit index lifecycle controls.

**Predict:** When the chunk size decreases, can the source `policy_id` disappear? It must not.

In [ ]:
for size, overlap in [(150, 25), (350, 35), (500, 50)]:
    chunks = chunk_policies(policies, size, overlap)
    print(f"{size} chars -> {len(chunks)} chunks; first: {chunks[0].metadata['chunk_id']}")
assert all("policy_id" in c.metadata and "chunk_id" in c.metadata
           for c in chunk_policies(policies, 150, 25))
store = build_store(policies, OfflineLexicalEmbeddings())
print("Chroma index built: yes (local, in-memory)")

## Role × evidence access mental model

Each row is a *trusted employee role*, each column a policy access group. The current demonstration implements `employee`, `risk`, `compliance`, and `admin` scopes. **Real identity must come from authenticated SSO**, not untrusted role fields in a browser request.

This figure is a policy matrix, not a cosine-similarity metric.

In [ ]:
import matplotlib.pyplot as plt
from RAG.enterprise.langchain_bfsi import ROLE_GROUPS
roles = ["employee", "risk", "compliance", "admin"]
groups = ["all", "risk", "compliance"]
matrix = [[int(g in ROLE_GROUPS[role]) for g in groups] for role in roles]
fig, ax = plt.subplots(figsize=(7, 3))
ax.imshow(matrix, vmin=0, vmax=1, cmap="Greys")
ax.set_xticks(range(len(groups)), groups)
ax.set_yticks(range(len(roles)), roles)
for i, role in enumerate(roles):
    for j, group in enumerate(groups):
        ax.text(j, i, "ALLOWED" if matrix[i][j] else "DENIED",
                ha="center", va="center", color="black" if matrix[i][j] == 0 else "white", fontsize=9)
ax.set_title("Department-level access groups (teaching simulation)")
plt.show()

## Retrieve as an employee and a compliance analyst

Try the same AML query under two principals. The query engine always pushes tenant, group and policy effective-date restrictions into Chroma's `filter` parameter. A risk analyst cannot retrieve `compliance`-labeled policy chunks by increasing `k` or asking a cleverly phrased question.

In [ ]:
query = "How fast must an AML suspicious transaction alert be triaged?"
for role in ("employee", "risk", "compliance", "admin"):
    principal = Principal("northstar-bank", role)
    hits = authorized_search(store, query, principal, TODAY, k=4)
    print(role, "->", [d.metadata["policy_id"] for d, _ in hits])
    assert all(d.metadata["access_group"] in ROLE_GROUPS[role] for d, _ in hits)
print("Filter for compliance:", scoped_filter(Principal("northstar-bank", "compliance"), TODAY))

## Tenant, stale-version and unknown-question regressions

A deprecated 2024 KYC policy must not silently win ranking in October 2026. The other tenant's `HARBOR-KYC-2026` is also forbidden regardless of a matching search phrase.

**Counterexample exercise:** demonstrate what can go wrong if you call unscoped `store.similarity_search` and pass the result to a model. That unscoped call is intentionally **not executed** here, to avoid teaching an insecure default.

In [ ]:
northstar = Principal("northstar-bank", "employee")
for question in (
    "Can I onboard a customer without address proof according to the expired 2024 KYC policy?",
    "What is the HarborCase identity process?",
    "Who is the CEO's favorite singer?",
):
    result = ask_policy(store, question, northstar, TODAY)
    print(question, "=>", result["mode"], "|", [x["policy_id"] for x in result["sources"]])
# Strong invariant: no unauthorized IDs ever reach the evidence payload.
restricted = authorized_search(store, "HarborCase Harbor Identity Desk KYC",
                               Principal("northstar-bank", "admin"), TODAY, max_l2_distance=2.1)
assert not any(d.metadata["tenant_id"] == "harbor-finance" for d, _ in restricted)
expired = authorized_search(store, "address proof could arrive later",
                            Principal("northstar-bank", "admin"), TODAY, max_l2_distance=2.1)
assert not any(d.metadata["policy_id"] == "KYC-2024" for d, _ in expired)

## How do we prove the search is working?

The next cell evaluates **document-level ranking**, not claim-level hallucination or regulatory accuracy. Negative questions include restricted, cross-tenant and unknown cases. A distance threshold must be calibrated separately for a semantic embedding model.

**Student insight:** strong retrieval recall does not prove answer faithfulness; strong permission filtering does not provide authentication by itself.

In [ ]:
evaluation = evaluate_access_and_retrieval(store, load_evaluation(), TODAY, k=3)
print(evaluation)
assert evaluation["answerable_n"] == 8
assert evaluation["restricted_or_unknown_n"] == 3

## Engineering checkpoint

1. Explain **pre-search authorization** versus post-search masking.
2. Show how a policy version is excluded by effective date.
3. Explain why a role copied from a JSON request is not a trusted identity.
4. Explain why a similarity distance of 1.4 is not 86% certainty.
5. State what is missing for production: server-side authentication, ACL revocation/reindexing, PII controls, audit logging, claim-level citation verification and legal validation.

Next: [07 · LangChain model orchestration](07_bfsi_langchain_end_to_end.ipynb).